# 02 · Process — Clean, validate, and build the analysis tables

Decisions come from [reports/02_prepare.md](../reports/02_prepare.md). The full write-up is [reports/03_process.md](../reports/03_process.md). The audit of Google's lab uses the lab's own table and lives in [notebook 03](03_analyze_lab_audit.ipynb).

```
ga_sessions_*  →  sql/process/01_sessions_clean.sql  →  sessions_clean.parquet
                        ├── remarketing_table  (D2 model: corrected)
                        └── touches / journeys (D1 attribution)
```

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import bq
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
from src.validate import validate_sessions
from src.tables import remarketing_table
from src.journeys import touches, journeys, revenue_cap

## 1. Clean session table

One row per session. It scans 0.81 GB the first time (about 8 minutes over REST) and reads from the cache after that. The SQL documents every cleaning rule.

In [2]:
sessions = bq.query("process/01_sessions_clean.sql", cache="sessions_clean")
sessions.shape

(903653, 32)

In [3]:
sessions.head(3).T

,0,1,2
session_key,0000538867824729259-1480506515-1480506515,000923747808925443-1483806859-1483806859,0023837072006818350-1481622746-1481622746
full_visitor_id,0000538867824729259,000923747808925443,0023837072006818350
visit_id,1480506515,1483806859,1481622746
visit_start_time,1480506515,1483806859,1481622746
session_start,2016-11-30 11:48:35+00:00,2017-01-07 16:34:19+00:00,2016-12-13 09:52:26+00:00
session_date,2016-11-30 00:00:00,2017-01-07 00:00:00,2016-12-13 00:00:00
visit_number,1,1,1
is_new_visit,True,True,True
channel,Direct,Social,Social
source,(direct),youtube.com,youtube.com


## 2. Validation against raw totals

Each check compares the extract with numbers measured on the raw BigQuery tables in Prepare.

In [4]:
checks = validate_sessions(sessions)
assert (checks.status == 'PASS').all(), checks[checks.status != 'PASS']
checks

,status,check,observed
0,PASS,Row count matches raw,"903,653"
1,PASS,session_key is unique,"903,653 unique"
2,PASS,Purchase sessions match raw,"11,552"
3,PASS,Revenue matches raw,"$1,780,149"
4,PASS,Internal visitors match Prepare,"37,332"
5,PASS,Internal sessions / purchases / revenue match ...,"76,536 / 5,408 / $730,376.50"
6,PASS,Internal flag is constant within each visitor,
7,PASS,No NULLs in engagement / outcome columns,
8,PASS,No negative values,
9,PASS,Date range is 2016-08-01 to 2017-08-01,2016-08-01 to 2017-08-01


## 3. GA already shifts credit to earlier campaigns

When a visitor returns directly, GA360 labels the visit with their *previous* campaign and sets `isTrueDirect`.

In [5]:
ext = sessions[~sessions.is_internal]
relabelled = ext[ext.is_true_direct & (ext.channel != "Direct")]
pd.Series({
    "external sessions": len(ext),
    "non-Direct label but actually a direct return": len(relabelled),
    "  share of external sessions": f"{len(relabelled)/len(ext):.1%}",
    "external purchases": int(ext.purchased.sum()),
    "  purchases on relabelled sessions": int(relabelled.purchased.sum()),
    "  share": f"{relabelled.purchased.sum()/ext.purchased.sum():.1%}",
})

external sessions                                827117
non-Direct label but actually a direct return     96809
  share of external sessions                      11.7%
external purchases                                 6144
  purchases on relabelled sessions                 1790
  share                                           29.1%
dtype: object

**Decision D-PR1:** journeys use the **arrival channel** (isTrueDirect → Direct). The last-click baseline keeps GA's labels, which is what the dashboards showed. A conservative relabel (only visits whose source reads `(direct)`) is kept for comparison; [notebook 05](05_analyze_attribution.ipynb) shows that form depends on the export day, not the visit.

## 4. D2 model table: corrected

External first-visit non-buyers. Label = purchase on a later visit within 30 days. Only first visits whose full 30-day window lies inside the data. Same train/test months as the lab.

In [6]:
rm = remarketing_table(sessions)
rm.to_parquet(ROOT / "data/processed/remarketing_table.parquet", index=False)
rm.groupby("split").agg(
    visitors=("buy_within_30d", "size"),
    buyers_30d=("buy_within_30d", "sum"),
    rate_pct=("buy_within_30d", lambda x: round(100 * x.mean(), 3)),
    revenue_30d_usd=("revenue_30d_usd", "sum"),
)

,visitors,buyers_30d,rate_pct,revenue_30d_usd
split,,,,
test,91431,323,0.353,53545.29
train,522599,1433,0.274,239574.97


## 5. D1 journey tables

A purchase or 30 days of inactivity ends a journey. 30-day lookback. Journeys ending 2016-08-31 – 2017-07-01. Revenue capped per purchase session at the 99th percentile ($1,606, set with the key account included). Google employees and the key account ([`src/segments.py`](../src/segments.py)) are left out of the journeys.

In [7]:
t = touches(sessions)
orders = t.loc[t.purchased, "revenue_usd"]
cap = revenue_cap(sessions)
j = journeys(t, revenue_cap_usd=cap)
t.to_parquet(ROOT / "data/processed/touches.parquet", index=False)
j.to_parquet(ROOT / "data/processed/journeys.parquet", index=False)

conv = j[j.converted]
assert len(conv) == len(orders), "every external purchase in the period must end exactly one journey"
pd.Series({
    "journeys": len(j),
    "converting journeys": len(conv),
    "converting journeys with >1 touch": f"{(conv.n_touches > 1).mean():.1%}",
    "arrival path differs from GA labels": f"{(conv.arrival_path != conv.ga_path).mean():.1%}",
    "revenue cap (p99 purchase session)": f"${cap:,.0f}",
    "purchase sessions above the cap: share of revenue": f"{orders[orders > cap].sum() / orders.sum():.1%}",
    "revenue raw / capped": f"${conv.revenue_usd.sum():,.0f} / ${conv.revenue_capped_usd.sum():,.0f}",
})

journeys                                                          580759
converting journeys                                                 5058
converting journeys with >1 touch                                  42.4%
arrival path differs from GA labels                                29.6%
revenue cap (p99 purchase session)                                $1,606
purchase sessions above the cap: share of revenue                  17.2%
revenue raw / capped                                 $721,771 / $664,824
dtype: object

In [8]:
conv.arrival_path.value_counts().head(10).rename("converting journeys")

arrival_path
Organic Search                                1389
Direct                                        1203
Organic Search > Direct                        443
Direct > Direct                                440
Paid Search                                    205
Direct > Direct > Direct                       187
Organic Search > Direct > Direct               183
Direct > Direct > Direct > Direct              109
Organic Search > Direct > Direct > Direct       78
Direct > Direct > Direct > Direct > Direct      57
Name: converting journeys, dtype: int64